In [1]:
import re
import torch
import pandas as pd 

In [2]:
torch.manual_seed(42)

In [3]:
data=pd.read_csv('football_qa_dataset_full.csv')

In [4]:
data

,question,answer
0,Who won the 1930 FIFA World Cup?,Uruguay
1,Which team was crowned champion of the 1930 FI...,Uruguay
2,Which country hosted the 1930 FIFA World Cup?,Uruguay
3,In which country was the 1930 FIFA World Cup h...,Uruguay
4,Who was the runner-up in the 1930 FIFA World Cup?,Argentina
...,...,...
2877,Which MLS club did Lionel Messi join in 2023?,Inter-Miami
2878,Which club signed Zinedine Zidane from Juventu...,Real-Madrid
2879,Which club signed Luis Figo directly from riva...,Real-Madrid
2880,Which club signed Diego Maradona from Barcelon...,Napoli


In [5]:
#tokenize

def tokenize(text):
    text=text.lower()
    # drop punctuation (?, commas, full stops, apostrophes...) but keep hyphens used in names like inter-miami
    text=re.sub(r"[^\w\s-]","",text)
    
    return text.split()

In [6]:
tokenize('Who won the 1930 FIFA World Cup?')

['who', 'won', 'the', '1930', 'fifa', 'world', 'cup']

In [7]:
#vocab
vocab={'<UNK>':0}

def build_vocab(row):
    token_question=tokenize(row['question'])
    token_answer=tokenize(row['answer'])
    
    merged_tokens=token_question + token_answer
    
    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)


In [8]:
data.apply(build_vocab,axis=1) #axis=1 means whole row 

0       None
1       None
2       None
3       None
4       None
        ... 
2877    None
2878    None
2879    None
2880    None
2881    None
Length: 2882, dtype: object

In [9]:
vocab

{'<UNK>': 0,
 'who': 1,
 'won': 2,
 'the': 3,
 '1930': 4,
 'fifa': 5,
 'world': 6,
 'cup': 7,
 'uruguay': 8,
 'which': 9,
 'team': 10,
 'was': 11,
 'crowned': 12,
 'champion': 13,
 'of': 14,
 'country': 15,
 'hosted': 16,
 'in': 17,
 'held': 18,
 'runner-up': 19,
 'argentina': 20,
 'lost': 21,
 'final': 22,
 'golden': 23,
 'boot': 24,
 'at': 25,
 'guillermo-stabile': 26,
 'top': 27,
 'scorer': 28,
 'how': 29,
 'many': 30,
 'goals': 31,
 'did': 32,
 'score': 33,
 '8': 34,
 '1934': 35,
 'italy': 36,
 'czechoslovakia': 37,
 'oldrich-nejedly': 38,
 '5': 39,
 '1938': 40,
 'france': 41,
 'hungary': 42,
 'leonidas': 43,
 '7': 44,
 '1950': 45,
 'brazil': 46,
 'ademir': 47,
 '9': 48,
 '1954': 49,
 'westgermany': 50,
 'switzerland': 51,
 'sandor-kocsis': 52,
 '11': 53,
 '1958': 54,
 'sweden': 55,
 'just-fontaine': 56,
 '13': 57,
 '1962': 58,
 'chile': 59,
 '1966': 60,
 'england': 61,
 'eusebio': 62,
 '1970': 63,
 'mexico': 64,
 'gerd-muller': 65,
 '10': 66,
 '1974': 67,
 'netherlands': 68,
 'grz

In [10]:
print(len(vocab))

2022


In [11]:
def text_to_indexs(text,vocab):
    indexed_text=[]
    
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
        
    return indexed_text

In [12]:
text_to_indexs('Which country hosted the 1930 FIFA World Cup?',vocab)

[9, 15, 16, 3, 4, 5, 6, 7]

In [13]:
len(vocab)

2022

In [14]:
from torch.utils.data import Dataset,DataLoader

In [15]:



class QADataset(Dataset):
    def __init__(self,data,vocab):
        self.data=data
        self.vocab=vocab
        
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, index):
        numerical_question=text_to_indexs(self.data.iloc[index]['question'],self.vocab)
        numerical_answer=text_to_indexs(self.data.iloc[index]['answer'],self.vocab)
        
        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [16]:
dataset=QADataset(data,vocab)

In [17]:
dataset[2]

(tensor([ 9, 15, 16,  3,  4,  5,  6,  7]), tensor([8]))

In [18]:
#train/test split
train_data=data.sample(frac=0.8,random_state=42)
test_data=data.drop(train_data.index)

train_dataset=QADataset(train_data,vocab)
test_dataset=QADataset(test_data,vocab)
print(len(train_dataset),len(test_dataset))

2306 576


In [19]:
dataloader=DataLoader(train_dataset,batch_size=1,shuffle=True)

In [20]:
question , answer = next(iter(dataloader))
print(question,answer[0])

tensor([[  9,  15,  32, 664, 665, 590]]) tensor([74])


In [21]:
import torch.nn as nn
vocab_size=len(vocab)

In [22]:
vocab_size

2022

In [23]:
dropout=0.3

In [24]:



class SImpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn=nn.RNN(50,128,batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc=nn.Linear(128,vocab_size)
        
    def forward(self,question):
        embedded_question=self.embedding(question)
        hidden,final=self.rnn(embedded_question)
        final=self.dropout(final.squeeze(0))
        output=self.fc(final)    
        
        return output

In [25]:
# learning_rate=0.002
epochs=30

In [26]:
model=SImpleRNN(len(vocab))

In [27]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=5e-4, weight_decay=1e-5)

In [28]:
def evaluate(model,dataset):
    model.eval()
    correct=0
    with torch.no_grad():
        for i in range(len(dataset)):
            question,answer=dataset[i]
            output=model(question.unsqueeze(0))
            correct+=int(output.argmax(dim=1).item()==answer[0].item())
    return correct/len(dataset)

In [29]:
#training loop

for epoch in range(epochs):
    model.train()
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        output=model(question)
        loss=criterion(output,answer[0])
        
        loss.backward()
        
        optimizer.step()
        
        total_loss=total_loss+loss.item()
    avg_loss=total_loss/len(dataloader)
    print(f'Epoch {epoch +1},loss:{avg_loss:4f},train acc:{evaluate(model,train_dataset):.3f},test acc:{evaluate(model,test_dataset):.3f}')

Epoch 1,loss:6.218558,train acc:0.118,test acc:0.083


Epoch 2,loss:4.902800,train acc:0.176,test acc:0.120


Epoch 3,loss:4.294886,train acc:0.203,test acc:0.116


Epoch 4,loss:3.863289,train acc:0.300,test acc:0.137


Epoch 5,loss:3.541082,train acc:0.341,test acc:0.134


...

Epoch 26,loss:0.448035,train acc:0.987,test acc:0.168


Epoch 27,loss:0.392362,train acc:0.984,test acc:0.161


Epoch 28,loss:0.366163,train acc:0.991,test acc:0.161


Epoch 29,loss:0.331647,train acc:0.995,test acc:0.163


Epoch 30,loss:0.296267,train acc:0.994,test acc:0.165


In [30]:
def predict(model,question,threshold=0.5):
    model.eval()
    
    #covert question to numbers
    numerical_question=text_to_indexs(question,vocab)
    
    #tensor
    question_tensor=torch.tensor(numerical_question).unsqueeze(0)
    
    #send to model
    with torch.no_grad():
        output=model(question_tensor)
    
    #conv=ert logits to proba 
    proba=torch.nn.functional.softmax(output,dim=1)
    
    #find max proba 
    value, index= torch.max(proba,dim=1)
    
    if value < threshold:
        print('idk')
    else:
        
        print(list(vocab.keys())[index])

In [31]:
predict(model,"team lost the final of 1954 FIFA World Cup?")

idk

In [32]:
predict(model,"Who won the 1954 FIFA World Cup?")

idk
